In [1]:
import ipywidgets as widgets
from IPython.display import display
import requests
import json
import os

DIRECTORIO_BASE = "/content/drive/MyDrive/data_tesis"
PROYECTO_GDC = "TCGA-KIRC"

# Se corrigió "Diagnostic Report" por "Pathology Report" en la jerarquía
jerarquia_gdc = {
    "Biospecimen": ["Slide Image", "Biospecimen Supplement"],
    "Clinical": ["Clinical Supplement", "Pathology Report"],
    "Transcriptome Profiling": ["Gene Expression Quantification", "miRNA Expression Quantification"],
    "Simple Nucleotide Variation": ["Masked Somatic Mutation"],
    "Copy Number Variation": ["Copy Number Segment", "Gene Level Copy Number"]
}

contenedor_principal = widgets.VBox()

def mostrar_paso_1():
    titulo = widgets.HTML("<b>Paso 1: Selecciona los requisitos de los pacientes (Filtro)</b>")
    diccionario_checks = {}
    hijos_acordeon = []
    titulos = []

    for categoria, tipos in jerarquia_gdc.items():
        checks = [widgets.Checkbox(description=t, value=False, indent=False) for t in tipos]
        diccionario_checks[categoria] = checks
        hijos_acordeon.append(widgets.VBox(checks))
        titulos.append(categoria)

    acordeon = widgets.Accordion(children=hijos_acordeon)
    for i, cat in enumerate(titulos):
        acordeon.set_title(i, cat)

    btn_siguiente = widgets.Button(description="Siguiente paso", button_style="info")
    mensaje_error = widgets.Output()

    def avanzar_paso_2(b):
        seleccionados = set()
        for checks in diccionario_checks.values():
            for chk in checks:
                if chk.value:
                    seleccionados.add(chk.description)

        if not seleccionados:
            with mensaje_error:
                print("Debes seleccionar al menos un requisito para continuar.")
            return

        mostrar_paso_2(seleccionados)

    btn_siguiente.on_click(avanzar_paso_2)
    contenedor_principal.children = [titulo, acordeon, btn_siguiente, mensaje_error]

def mostrar_paso_2(tipos_permitidos):
    titulo = widgets.HTML("<b>Paso 2: Selecciona los archivos a descargar</b>")
    diccionario_checks = {}
    hijos_acordeon = []
    titulos = []

    for categoria, tipos in jerarquia_gdc.items():
        tipos_filtrados = [t for t in tipos if t in tipos_permitidos]
        if not tipos_filtrados:
            continue

        checks = [widgets.Checkbox(description=t, value=False, indent=False) for t in tipos_filtrados]
        diccionario_checks[categoria] = checks
        hijos_acordeon.append(widgets.VBox(checks))
        titulos.append(categoria)

    acordeon = widgets.Accordion(children=hijos_acordeon)
    for i, cat in enumerate(titulos):
        acordeon.set_title(i, cat)

    btn_descargar = widgets.Button(description="Procesar y Descargar", button_style="success", icon="download")
    btn_volver = widgets.Button(description="Volver al Paso 1", button_style="warning")
    panel_botones = widgets.HBox([btn_volver, btn_descargar])
    salida_descarga = widgets.Output()

    def descargar(b):
        tipos_a_descargar = set()
        for checks in diccionario_checks.values():
            for chk in checks:
                if chk.value:
                    tipos_a_descargar.add(chk.description)

        if not tipos_a_descargar:
            with salida_descarga:
                print("Selecciona al menos un archivo para descargar.")
            return

        with salida_descarga:
            print(f"Buscando pacientes con: {', '.join(tipos_permitidos)}")

            endpoint_archivos = "https://api.gdc.cancer.gov/files"
            filtros = {
                "op": "and",
                "content": [
                    {"op": "in", "content": {"field": "cases.project.project_id", "value": [PROYECTO_GDC]}},
                    {"op": "in", "content": {"field": "data_type", "value": list(tipos_permitidos)}}
                ]
            }

            parametros = {
                "filters": json.dumps(filtros),
                "fields": "file_id,cases.case_id,data_type,file_name",
                "format": "JSON",
                "size": "10000"
            }

            respuesta = requests.get(endpoint_archivos, params=parametros)
            archivos_encontrados = respuesta.json()["data"]["hits"]

            pacientes = {}
            for archivo in archivos_encontrados:
                case_id = archivo["cases"][0]["case_id"]
                if case_id not in pacientes:
                    pacientes[case_id] = {"tipos_disponibles": set(), "archivos_descarga": []}

                pacientes[case_id]["tipos_disponibles"].add(archivo["data_type"])
                if archivo["data_type"] in tipos_a_descargar:
                    pacientes[case_id]["archivos_descarga"].append(archivo)

            archivos_finales = []
            for case_id, datos in pacientes.items():
                if tipos_permitidos.issubset(datos["tipos_disponibles"]):
                    archivos_finales.extend(datos["archivos_descarga"])

            print(f"Descargando {len(archivos_finales)} archivos...")

            for archivo in archivos_finales:
                tipo_dato_limpio = archivo["data_type"].replace(" ", "").lower()
                ruta_carpeta = os.path.join(DIRECTORIO_BASE, tipo_dato_limpio)
                os.makedirs(ruta_carpeta, exist_ok=True)

                ruta_archivo = os.path.join(ruta_carpeta, archivo["file_name"])

                if not os.path.exists(ruta_archivo):
                    res_descarga = requests.get(f"https://api.gdc.cancer.gov/data/{archivo['file_id']}", stream=True)
                    with open(ruta_archivo, "wb") as f:
                        for bloque in res_descarga.iter_content(chunk_size=8192):
                            if bloque:
                                f.write(bloque)
            print("Proceso completado exitosamente.")

    def volver(b):
        mostrar_paso_1()

    btn_descargar.on_click(descargar)
    btn_volver.on_click(volver)

    contenedor_principal.children = [titulo, acordeon, panel_botones, salida_descarga]

display(contenedor_principal)
mostrar_paso_1()

VBox()